# V11.12 Execution Evidence Companion

## tl;dr
No usable-Alpha certification. Buy-ticket constraints and scoped corporate-action gaps require physical-account validation.

## Context and methods
Replay the public aggregate audit,not a new market test. Two previously exposed2023–2024 accounts; counts are not independent samples. Detailed reconciliation lives in scripts/verify_execution_evidence.py with private frozen artifacts.

## Data
Run from the repository or notebooks directory. Only a public aggregate JSON is read; no warehouse access,credentials or sealed dates.

In [1]:
import json
from pathlib import Path
import duckdb
root = Path.cwd()
if not (root / 'docs/V11_12_RESULT.summary.json').exists():
    root = root.parent
source = root / 'docs/V11_12_RESULT.summary.json'
data = json.loads(source.read_text(encoding='utf-8'))
assert data['independent_audit_pass'] and not data['validated_alpha']
print('Source:', source.name, 'Trial lower bound:', data['raw_trial_lower_bound'])


Source: V11_12_RESULT.summary.json Trial lower bound: 3322


## Results
The denominator is original executed buy tickets for each frozen policy. Aggregate unallocated notional is not lost capital or return.

In [2]:
with duckdb.connect() as con:
    rows = con.execute("SELECT a.policy,a.buy_tickets,a.below_minimum_buy_tickets,a.unallocated_buy_fraction,a.extra_minimum_commission_cny FROM (SELECT unnest(rows) a FROM read_json_auto(?)) ORDER BY a.policy", [str(source)]).fetchall()
assert len(rows) == 2
assert sum(r[1] for r in rows) == 10311
for row in rows:
    print(row)
assert data['worklist_reason_counts'] == {'held_adjustment_change':477,'held_raw_missing':35}
print('Scoped keys:', data['scoped_worklist_keys'], 'stocks:', data['worklist_unique_instruments'])


('lowvol', 5484, 1980, 0.042173835453555006, 23739.12513424347)
('stable_lowrisk', 4827, 2111, 0.0476285632760697, 26998.518040150724)
Scoped keys: 512 stocks: 253


## Takeaways
Freeze the observation. Diagnose turnover/style on current data; use the scoped worklist for raw-share/cash reconciliation. No fabricated dividend events,no revised PnL and no Alpha Court PASS.

## QA boundary
Code cells were replayed in order in one standard Python namespace. Native Jupyter kernel and nbformat validation are unavailable in this host's installed runtimes; this is not a claim of kernel execution. Full engine tests and the independent audit were executed separately.